## Analysis of escape genes expression in GTEx

We see that expression of escape genes is different across tissues and potentially different tissues might be releasing these proteins into the blood stream. Here we take several tissues that are selected by high expression of Serpine1, and we plot how gene expressino in these tissues depend on the Age of the patient.

What we expect to see: trajectories should be consistent with UKBB trajectories (Age_Beta) values
What we wanna learn:
- what are the differences in expression patterns of escape genes between different organs (maybe in centerarians we just see the shift in secretion by specific organ)
- for the pathways: we have some genes among escape ones that are belonging to the same pathway, so are the trajectories of expression change with age the same for those genes? is pathway level escape pattern in centenarians is driven by 1/few genes from the pathway or it's combined effect? 

In [46]:
# Imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os  

PROJECT_ROOT = Path("/Users/achechenina/projects/centenerians")

In [ ]:
# Load GTEx data
file_path = Path(PROJECT_ROOT) / "data" / "GTEx" / "GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct"
gtex_expr = pd.read_csv(file_path, sep="\t", skiprows=2)
# saving as parquet
gtex_expr.to_pickle(Path(PROJECT_ROOT) / "data" / "GTEx" / "gtex_expr.pkl")
gtex_expr.head()

,Name,Description,GTEX-1117F-0005-SM-HL9SH,GTEX-1117F-0011-R10b-SM-GI4VE,GTEX-1117F-0011-R11b-SM-GIN8R,GTEX-1117F-0011-R2b-SM-GI4VL,GTEX-1117F-0011-R3a-SM-GJ3PJ,GTEX-1117F-0011-R4b-SM-GI4VM,GTEX-1117F-0011-R5a-SM-GI4VW,GTEX-1117F-0011-R6a-SM-GI4VX,...,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2326-SM-GOQYU,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2526-SM-GOQZ3,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
0,ENSG00000223972.5,DDX11L1,0.00000,0.000000,0.000000,0.00000,0.0000,0.012397,0.00000,0.029395,...,0.000000,0.0000,0.000000,0.00000,0.00000,0.000000,0.000000,0.017791,0.020221,0.025713
1,ENSG00000227232.5,WASH7P,1.33343,3.579280,10.189300,2.96650,3.6828,3.599210,3.51443,4.286380,...,5.464380,2.3599,2.427010,3.87837,1.85892,2.383760,2.886100,5.095810,0.842465,2.209520
2,ENSG00000278267.1,MIR6859-1,0.00000,0.000000,0.000000,0.00000,0.0000,0.000000,0.00000,0.000000,...,0.000000,0.0000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ENSG00000243485.5,MIR1302-2HG,0.00000,0.093825,0.034191,0.00000,0.0000,0.024750,0.00000,0.000000,...,0.062071,0.0000,0.086553,0.14685,0.00000,0.041073,0.053323,0.000000,0.000000,0.000000
4,ENSG00000237613.2,FAM138A,0.00000,0.000000,0.000000,0.01766,0.0000,0.000000,0.00000,0.000000,...,0.000000,0.0000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000


In [68]:
# loading specific organs expression data
# pathlib.Path.glob — stdlib glob.glob() expects a str, not Path (PosixPath is not subscriptable).
files = sorted(Path(PROJECT_ROOT).glob("data/GTEx/gene_tpm_v11_*.gct.gz"))
tissue_expression = {}

for f in files:
    tissue = os.path.basename(f).replace("gene_tpm_v11_", "").replace(".gct.gz", "")
    if tissue == 'kidney_medulla':
        continue    
    df = pd.read_csv(
        f,
        sep="\t",
        skiprows=2,
        compression="gzip"
    )
    df["tissue"] = tissue
    tissue_expression[tissue] = df

print(tissue_expression.keys())

dict_keys(['adipose_subcutaneous', 'adipose_visceral_omentum', 'artery_aorta', 'artery_coronary', 'artery_tibial', 'kidney_cortex', 'liver', 'whole_blood'])


In [69]:
tissue_expression['whole_blood'].head(3)

,Name,Description,GTEX-1117F-0005-SM-HL9SH,GTEX-111CU-0005-SM-GJ3PH,GTEX-111FC-0006-SM-H65Z1,GTEX-111YS-0006-SM-5NQBE,GTEX-1122O-0005-SM-5O99J,GTEX-1128S-0005-SM-5P9HI,GTEX-113IC-0006-SM-5NQ9C,GTEX-113JC-0006-SM-5O997,...,GTEX-ZXES-0005-SM-57WCB,GTEX-ZXG5-0005-SM-57WCN,GTEX-ZY6K-0006-SM-GRR18,GTEX-ZYFC-0006-SM-HL9SK,GTEX-ZYFD-0005-SM-HL9SL,GTEX-ZYT6-0006-SM-HL9SM,GTEX-ZYVF-0005-SM-HL9SN,GTEX-ZYW4-0006-SM-HAUXL,GTEX-ZZPU-0006-SM-HL9SO,tissue
0,ENSG00000290825.2,DDX11L16,0.0000,0.018842,0.0000,0.00000,0.00000,0.00000,0.00000,0.038768,...,0.00000,0.00000,0.032704,0.064996,0.00000,0.00000,0.00000,0.00000,0.00000,whole_blood
1,ENSG00000223972.6,DDX11L1,0.0000,0.000000,0.0000,0.00000,0.00000,0.00000,0.00000,0.000000,...,0.00000,0.00000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000,whole_blood
2,ENSG00000310526.1,WASH7P,1.1639,2.222020,2.3321,0.93515,2.28904,3.04628,7.91296,0.770171,...,3.22096,2.82263,4.969430,4.903870,2.26242,2.72374,1.34321,1.61599,2.20729,whole_blood


In [ ]:
# reding the Gtex metadata files
gtex_metadata = pd.read_csv(Path(PROJECT_ROOT) / "data" / "GTEx" / "Gtex_restricted.txt", sep="\t")
gtex_metadata.head()

,Tissue.Sample.ID,Tissue,Subject.ID,Sex,Age.Bracket,Hardy.Scale,Pathology.Categories,Pathology.Notes,url,AGE
0,GTEX-1117F-0126,Skin - Sun Exposed (Lower leg),GTEX-1117F,female,60-69,Slow death,NaN,"6 pieces, minimal fat, squamous epithelium is ...",https://brd.nci.nih.gov/brd/imagedownload/GTEX...,66.0
1,GTEX-1117F-0226,Adipose - Subcutaneous,GTEX-1117F,female,60-69,Slow death,NaN,"2 pieces, ~15% vessel stroma, rep delineated",https://brd.nci.nih.gov/brd/imagedownload/GTEX...,66.0
2,GTEX-1117F-0326,Nerve - Tibial,GTEX-1117F,female,60-69,Slow death,clean_specimens,"2 pieces, clean specimens",https://brd.nci.nih.gov/brd/imagedownload/GTEX...,66.0


In [70]:
# Use tissues_expr as main, and align to metadata and expression key names as needed
# Here we define a mapping between tissues_expr and gtex_metadata1['Tissue'] for clarity
tissue_label_mapping = {
    'adipose_subcutaneous': 'Adipose - Subcutaneous',
    'adipose_visceral_omentum': 'Adipose - Visceral (Omentum)',
    'artery_aorta': 'Artery - Aorta',
    'artery_coronary': 'Artery - Coronary',
    'artery_tibial': 'Artery - Tibial',
    'kidney_cortex': 'Kidney - Cortex',
    #'kidney_medulla': 'Kidney - Medulla',
    'liver': 'Liver',
    'whole_blood': 'Whole Blood',  # if present, else skip
}

for expr_name in tissue_expression.keys():
    patients = ['-'.join(i.split('-')[:2]) for i in tissue_expression[expr_name].columns]
    n_patients_in_meta = 0
    for patient in patients:
        if patient in gtex_metadata['SUBJID'].values:
            n_patients_in_meta += 1
    print(f"{expr_name}: {n_patients_in_meta} patients in metadata")


adipose_subcutaneous: 714 patients in metadata
adipose_visceral_omentum: 587 patients in metadata
artery_aorta: 472 patients in metadata
artery_coronary: 268 patients in metadata
artery_tibial: 691 patients in metadata
kidney_cortex: 104 patients in metadata
liver: 262 patients in metadata
whole_blood: 803 patients in metadata


In [71]:
# Load escape genes (CIAO×UKBB escape list fallback if CSV missing/empty)
_escape_csv_paths = [
    PROJECT_ROOT / "data" / "escape_genes.csv",
    PROJECT_ROOT / "sanju_version" / "2_results" / "escape_genes" / "escape_genes.csv",
]
escape_genes_33 = None
for _p in _escape_csv_paths:
    if _p.is_file():
        _df = pd.read_csv(_p)
        if len(_df) > 0:
            escape_genes_33 = _df
            break
if escape_genes_33 is None or len(escape_genes_33) == 0:
    _txt = PROJECT_ROOT / "results" / "data" / "escape_analysis" / "ciao_ukbb_escape_genes.txt"
    if not _txt.is_file():
        raise FileNotFoundError(
            "No escape gene list found. Add data/escape_genes.csv or results/data/escape_analysis/ciao_ukbb_escape_genes.txt"
        )
    _genes = [ln.strip() for ln in _txt.read_text().splitlines() if ln.strip()]
    escape_genes_33 = pd.DataFrame({"Gene": _genes})

_gene_col = "Gene" if "Gene" in escape_genes_33.columns else (
    "gene" if "gene" in escape_genes_33.columns else escape_genes_33.columns[0]
)
escape_gene_list = sorted(
    escape_genes_33[_gene_col].dropna().astype(str).str.strip().str.upper().unique().tolist()
)

_sanju_path = PROJECT_ROOT / "data" / "sanju_version" / "1_data" / "escape_genes_sanju_100.csv"
if _sanju_path.is_file():
    escape_genes_sanju_100 = pd.read_csv(_sanju_path)
else:
    escape_genes_sanju_100 = None

# --- Age vs expression: GTEx v11 tissue matrices × gtex_metadata (SUBJID, AGE) ---
SAMPLE_PREFIX = "GTEX-"


def _gtex_sample_columns(df: pd.DataFrame):
    return [c for c in df.columns if str(c).startswith(SAMPLE_PREFIX)]


def _gtex_sample_to_subjid(sample_id: str) -> str:
    """GTEX-1117F-0126-SM-... -> GTEX-1117F (matches gtex_metadata SUBJID)."""
    parts = str(sample_id).split("-")
    if len(parts) >= 2:
        return f"{parts[0]}-{parts[1]}"
    return str(sample_id)


def _gtex_organ_system(tissue_key: str) -> str:
    """Map v11 tissue file stem to organ system (for consistent colors)."""
    k = str(tissue_key).lower()
    if k.startswith("adipose"):
        return "adipose"
    if "artery" in k:
        return "artery"
    if k.startswith("kidney"):
        return "kidney"
    if k.startswith("liver"):
        return "liver"
    if "blood" in k:
        return "blood"
    return "other"


ORGAN_SYSTEM_COLORS = {
    "adipose": "#9b8ab8",
    "artery": "#c49a9a",
    "kidney": "#8aa3bf",
    "liver": "#95b89a",
    "blood": "#c4a882",
    "other": "#9a9a9a",
}

ORGAN_TITLE_COLORS = {
    "adipose": "#5c4d78",
    "artery": "#7a5558",
    "kidney": "#4d6278",
    "liver": "#556856",
    "blood": "#7a6648",
    "other": "#5a5a5a",
}


def _fmt_p(p) -> str:
    if p is None or (isinstance(p, float) and (np.isnan(p) or np.isinf(p))):
        return "—"
    if abs(p) < 1e-4:
        return f"{p:.1e}"
    return f"{p:.3g}"



def long_expr_age_for_gene(expr_wide: pd.DataFrame, meta_df: pd.DataFrame, gene_symbol: str) -> pd.DataFrame:
    """One row per RNA sample with finite AGE (from gtex_metadata) and TPM for gene."""
    if "SUBJID" not in meta_df.columns or "AGE" not in meta_df.columns:
        raise ValueError("meta_df must be gtex_metadata with SUBJID and AGE columns")
    g = gene_symbol.strip().upper()
    sub = expr_wide[expr_wide["Description"].astype(str).str.strip().str.upper() == g]
    if sub.empty:
        return pd.DataFrame(columns=["Tissue.Sample.ID", "tpm", "AGE"])
    cols = _gtex_sample_columns(expr_wide)
    if not cols:
        return pd.DataFrame(columns=["Tissue.Sample.ID", "tpm", "AGE"])
    tpm = pd.to_numeric(sub[cols].mean(axis=0), errors="coerce")
    out = pd.DataFrame({"Tissue.Sample.ID": tpm.index.astype(str), "tpm": tpm.to_numpy(dtype=float)})
    out["SUBJID"] = out["Tissue.Sample.ID"].map(_gtex_sample_to_subjid)
    m = meta_df[["SUBJID", "AGE"]].drop_duplicates(subset=["SUBJID"], keep="first").copy()
    m["SUBJID"] = m["SUBJID"].astype(str)
    m["AGE"] = pd.to_numeric(m["AGE"], errors="coerce")
    out = out.merge(m, on="SUBJID", how="left")
    out = out.drop(columns=["SUBJID"], errors="ignore")
    return out.dropna(subset=["AGE", "tpm"])


def plot_escape_age_expression_grid(
    gene_list: list,
    tissue_expression_dict: dict,
    meta_df: pd.DataFrame,
    *,
    min_n: int = 8,
    genes_per_page: int = 5,
    out_dir=None,
):
    """Tissues = columns, genes = rows; age on x, log2(TPM+1) on y. meta_df = gtex_metadata (SUBJID, AGE)."""
    from matplotlib.patches import Patch

    tissues_sorted = sorted(tissue_expression_dict.keys())
    n_tiss = len(tissues_sorted)
    out_dir = out_dir or (PROJECT_ROOT / "results" / "plots" / "GTEx")
    out_dir.mkdir(parents=True, exist_ok=True)

    sns.set_theme(style="ticks", context="paper", font_scale=1.0)
    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
            "axes.linewidth": 0.8,
            "xtick.major.width": 0.8,
            "ytick.major.width": 0.8,
            "axes.labelsize": 9,
            "axes.titlesize": 9,
            "xtick.labelsize": 7,
            "ytick.labelsize": 7,
        }
    )

    pages = []
    for start in range(0, len(gene_list), genes_per_page):
        pages.append((start, min(start + genes_per_page, len(gene_list))))

    saved = []
    for pi, (lo, hi) in enumerate(pages):
        genes_pg = gene_list[lo:hi]
        n_rows = len(genes_pg)
        fig_w, fig_h = 2.05 * n_tiss, 1.55 * n_rows
        fig, axes = plt.subplots(n_rows, n_tiss, figsize=(fig_w, fig_h), squeeze=False, sharex=True)

        for i, gene in enumerate(genes_pg):
            for j, tkey in enumerate(tissues_sorted):
                ax = axes[i, j]
                d = long_expr_age_for_gene(tissue_expression_dict[tkey], meta_df, gene)
                if len(d) == 0:
                    ax.set_axis_off()
                    ax.text(
                        0.5,
                        0.5,
                        f"{gene}\n{tkey}\nno expr / no age",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                        fontsize=7,
                        color="0.45",
                    )
                    continue

                x = d["AGE"].to_numpy(dtype=float)
                y = np.log2(np.clip(d["tpm"].to_numpy(dtype=float), 0, None) + 1.0)
                sys = _gtex_organ_system(tkey)
                c = ORGAN_SYSTEM_COLORS[sys]
                ax.scatter(
                    x,
                    y,
                    s=11,
                    alpha=0.55,
                    color=c,
                    edgecolors="0.98",
                    linewidths=0.2,
                    rasterized=True,
                    zorder=1,
                )
                if len(d) >= min_n and np.nanstd(x) > 1e-6:
                    coef = np.polyfit(x, y, 1)
                    xx = np.linspace(np.nanmin(x), np.nanmax(x), 40)
                    ax.plot(
                        xx,
                        np.poly1d(coef)(xx),
                        color="#1a1a1a",
                        lw=2.25,
                        alpha=0.92,
                        solid_capstyle="round",
                        zorder=3,
                    )
                stat_lines = [f"n = {len(d)}"]
                if len(d) >= 3:
                    try:
                        from scipy import stats as _st

                        rp, pp = _st.pearsonr(x, y)
                        rs, ps = _st.spearmanr(x, y)
                        stat_lines.append(f"r = {rp:.2f}  p = {_fmt_p(pp)}")
                        stat_lines.append(f"ρ = {rs:.2f}  p = {_fmt_p(ps)}")
                    except Exception:
                        pass
                ax.text(
                    0.03,
                    0.97,
                    "\n".join(stat_lines),
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=5.8,
                    color="0.2",
                    linespacing=1.15,
                    bbox=dict(boxstyle="round,pad=0.28", facecolor="white", edgecolor="0.78", linewidth=0.6, alpha=0.92),
                    zorder=5,
                )

                ax.set_xlim(18, 82)
                sns.despine(ax=ax, offset=1, trim=True)

                if i == n_rows - 1:
                    ax.set_xlabel("Age (years)")
                else:
                    ax.set_xlabel("")
                if j == 0:
                    ax.set_ylabel(f"{gene}\nlog$_2$(TPM+1)", fontsize=8)
                else:
                    ax.set_ylabel("")
                if i == 0:
                    ax.set_title(
                        tkey.replace("_", " "),
                        fontsize=8,
                        pad=4,
                        color=ORGAN_TITLE_COLORS[_gtex_organ_system(tkey)],
                        fontweight="600",
                    )

        fig.suptitle(
            f"GTEx expression vs donor age (escape genes; page {pi + 1}/{len(pages)})",
            fontsize=11,
            y=1.02,
        )
        legend_order = ["adipose", "artery", "kidney", "liver", "blood", "other"]
        handles = []
        present = {_gtex_organ_system(tk) for tk in tissues_sorted}
        for lab in legend_order:
            if lab in present:
                handles.append(
                    Patch(
                        facecolor=ORGAN_SYSTEM_COLORS[lab],
                        edgecolor="0.25",
                        linewidth=0.6,
                        label=lab.title(),
                    )
                )
        if handles:
            fig.legend(
                handles=handles,
                loc="upper center",
                bbox_to_anchor=(0.5, 0.995),
                ncol=min(len(handles), 6),
                frameon=False,
                fontsize=8,
                title="Organ system",
                title_fontsize=8,
            )
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        out_path = out_dir / f"escape_genes_age_expression_grid_p{pi + 1:02d}.pdf"
        fig.savefig(out_path, bbox_inches="tight", facecolor="white")
        fig.savefig(out_path.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        saved.append(out_path)

    print(f"Plotted {len(gene_list)} genes × {n_tiss} tissues (min_n={min_n}). Saved:")
    for p in saved:
        print(" ", p)
    return saved


# Requires: tissue_expression, gtex_metadata (SUBJID, AGE from Gtex_restricted.txt)
plot_escape_age_expression_grid(
    escape_gene_list,
    tissue_expression,
    gtex_metadata,
    min_n=8,
    genes_per_page=5,
)


Plotted 33 genes × 8 tissues (min_n=8). Saved:
  /Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p01.pdf
  /Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p02.pdf
  /Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p03.pdf
  /Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p04.pdf
  /Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p05.pdf
  /Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p06.pdf
  /Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p07.pdf


[PosixPath('/Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p01.pdf'),
 PosixPath('/Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p02.pdf'),
 PosixPath('/Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p03.pdf'),
 PosixPath('/Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p04.pdf'),
 PosixPath('/Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p05.pdf'),
 PosixPath('/Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p06.pdf'),
 PosixPath('/Users/achechenina/projects/centenerians/results/plots/GTEx/escape_genes_age_expression_grid_p07.pdf')]

Interestingly, we see very strong directionality in adipose tissue for those genes, and also in whole blood. However, usually in liver there is no strong trends.